# **Credit Risk Prediction System**

---

## **1. Objective**
The objective of this project is to develop a **Credit Risk Prediction System** that estimates the  
**Probability of Default (PD)** for loan applicants **prior to loan approval**.

Rather than producing a binary accept/reject decision, the model is designed to:
- Support **risk-based lending**
- Enable **portfolio-level risk control**
- Assist in **expected loss minimization**

---

## **2. Target Variable Definition**

**Target Variable:** `good_bad`

| Value | Loan Outcome |
|------:|-------------|
| **1** | **Good Loan** - Fully Paid or Currently Active |
| **0** | **Bad Loan** - Defaulted, Charged-Off, or Severely Delinquent |

The target variable captures the **observed credit performance** of a loan over its lifecycle.

---

## **3. Machine Learning Task**

- **Problem Type:** Binary Classification  
- **Data Characteristics:** Strong class imbalance  
- **Prediction Objective:** **Probability of Default (PD)**  
- **Model Output:** Continuous risk score in the range **[0, 1]**

The system is explicitly designed for **risk ranking and prioritization**, not hard classification.

---

## **4. Evaluation Philosophy**

Model evaluation emphasizes **ranking quality and risk separation**, consistent with  
real-world credit risk management practices.

### **4.1 Primary Metric**
- **ROC–AUC (Receiver Operating Characteristic – Area Under Curve)**  
  Measures the model’s ability to rank higher-risk borrowers ahead of lower-risk borrowers.

### **4.2 Secondary Metrics**
- **Recall (Bad Loans):** Proportion of actual bad loans correctly identified  
- **KS Statistic:** Maximum separation between cumulative distributions of good and bad loans

### **4.3 Excluded Metric**
- **Accuracy** is deliberately excluded due to severe class imbalance and limited business relevance.

---

## **5. Modeling Principles**

The model is developed with the following principles:
- **Probability-based risk estimation**
- **Policy-driven cutoff selection**
- **Expected loss minimization**
- **Portfolio-level risk control**
- **Interpretability and stability** suitable for financial decision-making

---
---

## ***1. Importing Required Libraries***

In [15]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve
)
from sklearn.model_selection import train_test_split   
from sklearn.linear_model import LogisticRegression

## ***2. Loading ML Ready Dataset***

In [4]:
df_ml = pd.read_csv(r"C:\Users\tejas\OneDrive\Desktop\CPMLF Capstone Projects\CREDIT CARD RISK\data\processed\loan_data_feature_engineered.csv")

In [5]:
df_ml

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,log_tot_cur_bal,monthly_income,emi_to_income,revol_util_frac,acc_utilization,credit_age_years,has_delinquency,has_public_record,emp_length_bucket,rate_per_year
0,1077501,1296599,5000,5000,4975.0,0,10.65,162.87,1,6,...,NaN,2000.000000,0.081435,0.837,0.333333,32.916667,0,0,high,31.95
1,1077430,1314167,2500,2500,2500.0,1,15.27,59.83,2,13,...,NaN,2500.000000,0.023932,0.094,0.750000,18.666667,0,0,very_low,76.35
2,1077175,1313524,2400,2400,2400.0,0,15.96,84.33,2,14,...,NaN,1021.000000,0.082595,0.985,0.200000,16.083333,0,0,high,47.88
3,1076863,1277178,10000,10000,10000.0,0,13.49,339.31,2,10,...,NaN,4100.000000,0.082759,0.210,0.270270,21.833333,0,0,high,40.47
4,1075358,1311748,3000,3000,3000.0,1,12.69,67.79,1,9,...,NaN,6666.666667,0.010169,0.539,0.394737,21.916667,0,0,very_low,63.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
466280,8598660,1440975,18400,18400,18400.0,1,14.47,432.64,2,11,...,12.594727,9166.666667,0.047197,0.776,0.500000,14.666667,0,0,mid,72.35
466281,9684700,11536848,22000,22000,22000.0,1,19.97,582.50,3,19,...,12.309671,6500.000000,0.089615,0.463,0.600000,20.500000,0,1,high,99.85
466282,9584776,11436914,20700,20700,20700.0,1,16.99,514.34,3,15,...,11.206387,3833.333333,0.134176,0.511,0.418605,16.000000,0,0,high,84.95
466283,9604874,11457002,2000,2000,2000.0,0,7.90,62.59,0,3,...,13.290605,6916.666667,0.009049,0.215,0.777778,14.833333,1,0,low,23.70


In [6]:
print("Dataset Shape: ", df_ml.shape)

Dataset Shape:  (466285, 202)


In [7]:
df_ml.head(10)

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,log_tot_cur_bal,monthly_income,emi_to_income,revol_util_frac,acc_utilization,credit_age_years,has_delinquency,has_public_record,emp_length_bucket,rate_per_year
0,1077501,1296599,5000,5000,4975.0,0,10.65,162.87,1,6,...,NaN,2000.000000,0.081435,0.837,0.333333,32.916667,0,0,high,31.95
1,1077430,1314167,2500,2500,2500.0,1,15.27,59.83,2,13,...,NaN,2500.000000,0.023932,0.094,0.750000,18.666667,0,0,very_low,76.35
2,1077175,1313524,2400,2400,2400.0,0,15.96,84.33,2,14,...,NaN,1021.000000,0.082595,0.985,0.200000,16.083333,0,0,high,47.88
3,1076863,1277178,10000,10000,10000.0,0,13.49,339.31,2,10,...,NaN,4100.000000,0.082759,0.210,0.270270,21.833333,0,0,high,40.47
4,1075358,1311748,3000,3000,3000.0,1,12.69,67.79,1,9,...,NaN,6666.666667,0.010169,0.539,0.394737,21.916667,0,0,very_low,63.45
5,1075269,1311441,5000,5000,5000.0,0,7.90,156.46,0,3,...,NaN,3000.000000,0.052153,0.283,0.750000,13.083333,0,0,low,23.70
6,1069639,1304742,7000,7000,7000.0,1,15.96,170.08,2,14,...,NaN,3917.000000,0.043421,0.856,0.636364,12.416667,0,0,high,79.80
7,1072053,1288686,3000,3000,3000.0,0,18.64,109.43,4,20,...,NaN,4000.000000,0.027357,0.875,1.000000,10.916667,0,0,high,55.92
8,1071795,1306957,5600,5600,5600.0,1,21.28,152.39,5,26,...,NaN,3333.333333,0.045717,0.326,0.846154,13.666667,0,0,mid,106.40
9,1071570,1306721,5375,5375,5350.0,1,12.69,121.45,1,9,...,NaN,1250.000000,0.097160,0.365,0.666667,13.250000,0,0,very_low,63.45


## ***3. Target Distribution***

In [8]:
target_col = "good_bad"

df_ml[target_col].value_counts()
df_ml[target_col].value_counts(normalize=True)

good_bad
1    0.890693
0    0.109307
Name: proportion, dtype: float64

In [9]:
display(Markdown("""
### ***Target Variable Distribution Insight***
---
Loan default datasets are **naturally imbalanced**.

A naive model that predicts **“good” for every customer** can still achieve **high accuracy**,  
but such a model is **useless for real-world credit risk control**.

Therefore, instead of relying on accuracy,  
we must focus on **probability-ranking metrics** such as:
- ROC-AUC
- KS Statistic
- Precision-Recall

These metrics help evaluate how well the model **separates good vs bad borrowers**.
"""))


### ***Target Variable Distribution Insight***
---
Loan default datasets are **naturally imbalanced**.

A naive model that predicts **“good” for every customer** can still achieve **high accuracy**,  
but such a model is **useless for real-world credit risk control**.

Therefore, instead of relying on accuracy,  
we must focus on **probability-ranking metrics** such as:
- ROC-AUC
- KS Statistic
- Precision-Recall

These metrics help evaluate how well the model **separates good vs bad borrowers**.


## ***4. Target Vs Features***

In [10]:
from IPython.display import display, Markdown

corr_with_target = (
    df_ml.corr(numeric_only=True)[target_col]
    .sort_values(ascending=False)
)

# Show correlation output
display(corr_with_target.head(15))

# Show explanation just below the result
display(Markdown("""
### Correlation Sanity Check – Feature Leakage Validation

- No **post-loan payment variables** appear among top correlated features  
- No **outcome-encoded or target-leaking variables** remain  
- The feature set strictly represents **pre-decision borrower information**  

This confirms the dataset is **safe for predictive modeling** and suitable for
real-world credit risk scorecard development.
"""))


good_bad                  1.000000
loan_status:Current       0.337164
loan_status:Fully Paid    0.283768
out_prncp                 0.158676
out_prncp_inv             0.158662
grade:A                   0.098972
member_id                 0.096759
id                        0.095883
issue_d_date              0.087575
log_annual_inc            0.070266
grade:B                   0.063509
tot_cur_bal               0.051197
monthly_income            0.049864
annual_inc                0.049864
initial_list_status       0.048032
Name: good_bad, dtype: float64


### Correlation Sanity Check – Feature Leakage Validation

- No **post-loan payment variables** appear among top correlated features  
- No **outcome-encoded or target-leaking variables** remain  
- The feature set strictly represents **pre-decision borrower information**  

This confirms the dataset is **safe for predictive modeling** and suitable for
real-world credit risk scorecard development.


## ***5. Feature Matrix Finalization***

In [11]:
# Separating features and target
X = df_ml.drop(columns=[target_col])
y = df_ml[target_col]

print("Feature matrix shape:", X.shape)

display(Markdown("""
### ***Feature–Target Separation (Banking ML Context)***

- `X` contains only **predictor variables** available **before loan approval**
- `y` represents the **default outcome** (Good / Bad)

This separation ensures the model mimics a **real credit decision environment**,  
where outcomes are **unknown at decision time**.
"""))

Feature matrix shape: (466285, 201)



### ***Feature–Target Separation (Banking ML Context)***

- `X` contains only **predictor variables** available **before loan approval**
- `y` represents the **default outcome** (Good / Bad)

This separation ensures the model mimics a **real credit decision environment**,  
where outcomes are **unknown at decision time**.


## ***6. Train-Test Split***

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

Train shape: (373028, 201)
Test shape : (93257, 201)
